# Student Task: Prompt Chaining for Story Writing

## Goal
In this task, you will build a short science-fiction story using prompt chaining and iterative generation.

You will guide the model through these stages:

```text
Premise -> Outline -> Opening -> Continuation -> Final story
```

Some code is already completed for you. Complete every section marked `TODO`.

## Learning objectives

By the end of this task, you should be able to:

- Explain how prompt chaining passes output from one prompt into the next.
- Use a persona, context, constraints, and output instructions in a prompt.
- Continue a long generation over multiple model calls.
- Add a stopping condition to a generation loop.
- Count and inspect the final output.

## Task requirements

Your final program should:

1. Generate a one-sentence premise about a lost city on Mars.
2. Generate a story outline using the premise.
3. Generate the opening section using the premise and outline.
4. Continue the story at least twice.
5. Stop when the model writes `IAMDONE` or when the maximum number of continuations is reached.
6. Print the final story and its word count.

Do not place your API key directly in the notebook.

## 1. Setup

In [7]:
!pip install -q -U openai python-dotenv

import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
api_key = os.getenv("OPENROUTER_API_KEY")
client = OpenAI(api_key=api_key, base_url="https://openrouter.ai/api/v1")
MODEL = "~deepseek/deepseek-v4-flash-latest"

print("Setup complete")

Setup complete


## 2. Shared writing instructions

The variables below are completed. You may improve the writing guidelines after the main task works.

In [8]:
persona = "You are a creative science-fiction author writing for a general audience."

guidelines = """
Write vivid scenes with sensory details.
Develop the characters' goals and conflicts.
Do not summarize the story too quickly.
Continue naturally from the existing draft.
"""

print(persona)

You are a creative science-fiction author writing for a general audience.


## 3. Build the prompt chain

The first prompt is provided as an example. Complete the outline, opening, and continuation prompts.

Remember: `{{premise}}`, `{{outline}}`, and `{{story_text}}` are placeholders. They will be filled later using `.format()`.

In [9]:
premise_prompt = f"""
{persona}

Write one exciting sentence for a science-fiction story about a lost city on Mars.
"""

# TODO 1: Write outline_prompt using the premise placeholder.
# Ask the model for 5-7 major plot points.
outline_prompt = f"""
{persona}

You have a gripping premise in mind:

{{premise}}

Write an outline for the plot of your story. List 5-7 major plot points.
"""

# TODO 2: Write starting_prompt using the premise and outline placeholders.
# Ask for 500-800 words and introduce at least one important character.
starting_prompt = f"""
{persona}

You have a gripping premise in mind:

{{premise}}

Your outline for the story is:

{{outline}}

Write the opening section of the story. Write between 500 and 800 words and
introduce at least one important character.

{guidelines}
"""

# TODO 3: Write continuation_prompt using premise, outline, and story_text.
# Ask the model to continue the story and write IAMDONE when completely finished.
continuation_prompt = f"""
{persona}

You have a gripping premise in mind:

{{premise}}

Your outline for the story is:

{{outline}}

Here is the story so far:

{{story_text}}

Continue the story from where it left off. Once the story is completely
finished, write IAMDONE.

{guidelines}
"""

## 4. Generate the premise

The first model call creates the one-sentence story premise.

In [10]:
# TODO 4: Generate the outline with outline_prompt.format(premise=premise).
# Save the model output in outline and print it.
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": premise_prompt}],
)

premise = response.choices[0].message.content.strip()

print("Premise:")
print(premise)

Premise:
Beneath the rust-red dunes, the lost city of Mars did not sleep—it dreamed in electric pulses, its ancient machines waking one by one to the rhythm of our approaching footsteps.


## 5. Generate the outline

The outline prompt receives the generated premise and asks the model to plan the major plot points.

In [11]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": outline_prompt.format(premise=premise)}],
)

outline = response.choices[0].message.content.strip()
print("Outline:")
print(outline)

Outline:
**Title:** *The Dream Circuit*

**Major Plot Points:**

1. **The Signal in the Sand**  
A small survey crew on Mars detects an impossible electromagnetic rhythm pulsing from beneath the rust-red dunes—a pattern that looks like a heartbeat. They hike toward the source, and with every footstep, new lights flicker awake beneath the sand. The lost city is not dead; it is dreaming, and their approach is slowly pulling it into consciousness.

2. **Descent into the Dreaming City**  
The crew breaks into a buried cathedral of alien machines. Holograms shimmer with the images of vanished Martians, ancient engines begin to hum, and maintenance drones stir from millennia of sleep. They realize the city is *interpreting* them—not as intruders, but as long-awaited dreamers returning to finish a ritual. But not every part of the city is welcoming.

3. **The City Turns Hostile**  
A deep-security system activates, convinced the crew are impurities in the dream. Corridors reconfigure, gravity

## 6. Generate the opening

The opening prompt receives both the premise and the outline, then asks the model to begin the story.

In [12]:
# TODO 5: Generate the opening with starting_prompt.format(...).
# Save the model output in starting_draft and print it.
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": starting_prompt.format(premise=premise, outline=outline)}],
)

starting_draft = response.choices[0].message.content.strip()

print("Opening:")
print(starting_draft)

Opening:
The sensor读数 was impossible. Mira Voss stared at the data stream on her wrist console, the lines jittering with a pattern that spoke of rhythm, not random electro-magnetic noise. It was a thrum, a bass-note pulse, rising from beneath two hundred meters of rust-red basalt.

Mira had spent a decade of her life hunting for the fossilized whispers of alien life: isotopic traces in Martian regolith, microscopic crystal formations that hinted at ancient water. She had never expected to find a heartbeat.

"Status report, Voss. You've gone quiet," a voice crackled in her ear. Osei. His tone was always probing, searching for a flaw in her analysis.

She squinted against the lowering sun, its light a wan saffron disk in the pink sky. "There's a signal, Osei. It's... coherent. Periodic. It's pulsing."

"Geothermal harmonics can create pseudo-cycles," he countered. "You know that."

"Not like this. It's decelerating. A true curve, not a sine wave."

Her boots crunched on the loose regolit

## 7. Continue the story once

The first continuation is generated manually so the process can be inspected before the iterative loop runs.

In [13]:
draft = starting_draft

# TODO 6: Call the model with continuation_prompt.format(...).
# Use premise, outline, and story_text=draft.
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": continuation_prompt.format(premise=premise, outline=outline, story_text=draft)}],
)

continuation = response.choices[0].message.content.strip()

print("Continuation:")
print(continuation)

Continuation:
The archway had been sealed for three seconds before Mira’s pulse caught up with her ears. She slammed her glove against the basalt—no seam, no join, as if the rock had grown whole around them. “Osei, we’re sealed in.”

“I noticed.” His voice came breathless through the comm. Behind her, Anja, the mission’s biochemist, had shouldered off her pack and was pressing a sensor against the wall. “No residual radiation. It’s not a blast door—it’s smart material. It wants us inside.”

“Wants?” Mira repeated. She turned. The corridor ahead was high and dark, but the luminous panels along the ceiling had begun to glow—not electric, but pale, like moonstones softening in time to a distant heartbeat.

“Or it just doesn’t want us out,” Osei said.

They had no choice. They walked.

The corridor sloped downward. With every step, the glow deepened, and the air grew thick with a scent Mira couldn’t place—not dust, not metal, but something green and alive, like rain on a planet that had ne

## 8. Build the iterative loop

The loop appends each continuation to the draft and stops when the model writes `IAMDONE` or the maximum number of calls is reached.

In [14]:
# TODO 7: Add the first continuation to draft.
draft = draft + "\n\n" + continuation

# TODO 8: Set a maximum number of additional calls.
MAX_CONTINUATIONS = 3

for _ in range(MAX_CONTINUATIONS):
    # TODO 9: Stop if IAMDONE appears in continuation.
    if "IAMDONE" in continuation:
        break

    # TODO 10: Request the next continuation.
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": continuation_prompt.format(premise=premise, outline=outline, story_text=draft)}],
    )

    continuation = response.choices[0].message.content.strip()

    # TODO 11: Append the new continuation to draft.
    draft = draft + "\n\n" + continuation

# TODO 12: Remove IAMDONE and save the cleaned story in final.
final = draft.replace("IAMDONE", "").strip()
print(final)

The sensor读数 was impossible. Mira Voss stared at the data stream on her wrist console, the lines jittering with a pattern that spoke of rhythm, not random electro-magnetic noise. It was a thrum, a bass-note pulse, rising from beneath two hundred meters of rust-red basalt.

Mira had spent a decade of her life hunting for the fossilized whispers of alien life: isotopic traces in Martian regolith, microscopic crystal formations that hinted at ancient water. She had never expected to find a heartbeat.

"Status report, Voss. You've gone quiet," a voice crackled in her ear. Osei. His tone was always probing, searching for a flaw in her analysis.

She squinted against the lowering sun, its light a wan saffron disk in the pink sky. "There's a signal, Osei. It's... coherent. Periodic. It's pulsing."

"Geothermal harmonics can create pseudo-cycles," he countered. "You know that."

"Not like this. It's decelerating. A true curve, not a sine wave."

Her boots crunched on the loose regolith as she 

## 9. Evaluate the result

The final output is printed, its word count is calculated, and the completion marker is checked.

In [15]:
# TODO 13: Count the words in final.
word_count = len(final.split())
print(f"Final story word count: {word_count}")

# Extra check added for this student task: verify that the completion marker is removed.
print("Completion marker removed:", "IAMDONE" not in final)

Final story word count: 2380
Completion marker removed: True


## Submission checklist

- [ ] All `TODO` sections are completed.
- [ ] The notebook runs from top to bottom without errors.
- [ ] The premise, outline, opening, and final story are printed.
- [ ] The final story contains at least two continuations.
- [ ] The word count is printed.
- [ ] Reflection questions are answered.
- [ ] No API key is written directly in the notebook.